# ATS - Ollama on Colab T4 GPU

**Step 0:** Runtime > Change runtime type > **T4 GPU** > Save

Then run all cells top to bottom.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("\nIf you see Tesla T4 above, you're good. Otherwise: Runtime > Change runtime type > T4 GPU")

In [ ]:
# Cell 2: Install everything
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils lshw > /dev/null 2>&1
print("1/3 System deps installed")

!curl -fsSL https://ollama.com/install.sh | sh
print("\n2/3 Ollama installed")

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("3/3 Cloudflared installed")

print("\nAll installed!")

In [ ]:
# Cell 3: Start Ollama server (background)
import subprocess, time, os, urllib.request, json

# Kill old processes
!pkill -f ollama 2>/dev/null; pkill -f cloudflared 2>/dev/null
time.sleep(2)

# Start Ollama with nohup so it survives
!nohup env OLLAMA_HOST=0.0.0.0:11434 /usr/local/bin/ollama serve > /tmp/ollama.log 2>&1 &
time.sleep(5)

# Verify
try:
    resp = urllib.request.urlopen("http://localhost:11434", timeout=5)
    print("Ollama server: RUNNING")
except:
    print("Ollama server: NOT RUNNING - check /tmp/ollama.log")
    !cat /tmp/ollama.log | tail -5

In [ ]:
# Cell 4: Pull model
!ollama pull gemma2:2b
print("\nModel ready!")
print("\nGPU Memory:")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader

In [ ]:
# Cell 5: Speed test (warm up model on GPU)
import time, json, urllib.request

# Warm up
print("Loading model into GPU memory...")
warm = json.dumps({"model":"gemma2:2b","prompt":"hi","stream":False,"options":{"num_predict":1},"keep_alive":"30m"}).encode()
req = urllib.request.Request("http://localhost:11434/api/generate", data=warm, headers={"Content-Type":"application/json"})
urllib.request.urlopen(req, timeout=120)
print("Model loaded!\n")

# Speed test
payload = json.dumps({"model":"gemma2:2b","prompt":'Return JSON: {"status":"ok"}',"stream":False,"options":{"num_predict":20,"temperature":0},"keep_alive":"30m"}).encode()
req = urllib.request.Request("http://localhost:11434/api/generate", data=payload, headers={"Content-Type":"application/json"})
start = time.time()
resp = urllib.request.urlopen(req, timeout=60)
result = json.loads(resp.read())
elapsed = time.time() - start

print(f"Response: {result.get('response','')}")
print(f"Time: {elapsed:.2f}s")
if elapsed < 10:
    print(f"\nGPU is working! ({elapsed:.1f}s)")
else:
    print(f"\nSLOW ({elapsed:.1f}s) - GPU not active")

In [ ]:
# Cell 6: Start tunnel (background) and get URL
import subprocess, time, re

!pkill -f cloudflared 2>/dev/null
time.sleep(1)

# Start cloudflared in background, log to file
!nohup /usr/local/bin/cloudflared tunnel --url http://localhost:11434 > /tmp/tunnel.log 2>&1 &

# Wait for tunnel URL to appear in log
print("Starting tunnel...")
tunnel_url = None
for i in range(30):
    time.sleep(2)
    try:
        with open("/tmp/tunnel.log", "r") as f:
            log = f.read()
        # Look for https://SOMETHING.trycloudflare.com (must have subdomain)
        for line in log.splitlines():
            idx = line.find("https://")
            if idx != -1 and "trycloudflare.com" in line[idx:]:
                candidate = line[idx:].split()[0].strip()
                # Must have subdomain (not just https://trycloudflare.com)
                if candidate.count(".") >= 2 and candidate.endswith(".com"):
                    tunnel_url = candidate
                    break
        if tunnel_url:
            break
    except:
        pass

if tunnel_url:
    print("\n" + "="*60)
    print("TUNNEL URL: " + tunnel_url)
    print("="*60)
    print("\nPaste this in your .env file:")
    print("OLLAMA_BASE_URL=" + tunnel_url)
    print("="*60)
else:
    print("Tunnel URL not found automatically. Looking in log...\n")
    !cat /tmp/tunnel.log
    print("\nFind the line with https://xxxx.trycloudflare.com and copy that URL")

In [ ]:
# Cell 7: VERIFY tunnel works end-to-end (IMPORTANT - run this before using backend!)
import urllib.request, json, time

# Read tunnel URL from log
tunnel_url = None
with open("/tmp/tunnel.log", "r") as f:
    for line in f.read().splitlines():
        idx = line.find("https://")
        if idx != -1 and "trycloudflare.com" in line[idx:]:
            candidate = line[idx:].split()[0].strip()
            if candidate.count(".") >= 2 and candidate.endswith(".com"):
                tunnel_url = candidate
                break

if not tunnel_url:
    print("No tunnel URL found! Re-run Cell 6.")
else:
    print(f"Testing tunnel: {tunnel_url}")

    # Test 1: Basic connectivity
    print("\n1. Basic connectivity...")
    try:
        resp = urllib.request.urlopen(tunnel_url, timeout=10)
        print("   OK - tunnel reaches Ollama")
    except Exception as e:
        print(f"   FAILED: {e}")

    # Test 2: Actual model request through tunnel
    print("\n2. Model request through tunnel...")
    try:
        payload = json.dumps({"model":"gemma2:2b","prompt":'Say ok',"stream":False,"options":{"num_predict":5},"keep_alive":"30m"}).encode()
        req = urllib.request.Request(tunnel_url + "/api/generate", data=payload, headers={"Content-Type":"application/json"})
        start = time.time()
        resp = urllib.request.urlopen(req, timeout=30)
        result = json.loads(resp.read())
        elapsed = time.time() - start
        print(f"   OK - Response: {result.get('response','')} ({elapsed:.1f}s)")
    except Exception as e:
        print(f"   FAILED: {e}")

    # Test 3: Chat endpoint (same as your backend uses)
    print("\n3. Chat endpoint (same as backend uses)...")
    try:
        payload = json.dumps({"model":"gemma2:2b","messages":[{"role":"user","content":"Say hello"}],"stream":False,"options":{"num_predict":5},"keep_alive":"30m"}).encode()
        req = urllib.request.Request(tunnel_url + "/api/chat", data=payload, headers={"Content-Type":"application/json"})
        start = time.time()
        resp = urllib.request.urlopen(req, timeout=30)
        result = json.loads(resp.read())
        elapsed = time.time() - start
        content = result.get("message",{}).get("content","")
        print(f"   OK - Response: {content} ({elapsed:.1f}s)")
    except Exception as e:
        print(f"   FAILED: {e}")

    print("\n" + "="*60)
    print("If all 3 tests show OK:")
    print(f"  OLLAMA_BASE_URL={tunnel_url}")
    print("Paste above in your .env and start your backend!")
    print("="*60)

In [ ]:
# Cell 8: Keep session alive (run this and leave it)
# This prevents Colab from disconnecting due to inactivity
import time
print("Keeping session alive... Don't close this tab.")
print("Your tunnel and Ollama are running in background.")
print("\nPress the STOP button when you're done.")
try:
    i = 0
    while True:
        time.sleep(60)
        i += 1
        if i % 5 == 0:
            print(f"Still alive... ({i} min)")
except KeyboardInterrupt:
    print("Session ended.")